# Iceberg Metadata & Statistics via Trino

This notebook queries Iceberg metadata tables through **Trino** (not DuckDB).  
Trino exposes Iceberg hidden metadata tables using the `$` suffix — the table name and suffix must be quoted together as a single identifier:

```sql
-- correct
SELECT * FROM iceberg.demo."transactions$history"

-- wrong — $ is not a valid identifier character outside quotes
SELECT * FROM iceberg.demo.transactions.$history

```

For more details about availble metadata tables in trino **check** the **[link](https://trino.io/docs/current/connector/iceberg.html#metadata-tables)**

| Metadata Table | What it shows |
|---|---|
| `"table$history"` | Every snapshot commit — lineage |
| `"table$snapshots"` | Snapshot details: operation, file/record counts |
| `"table$files"` | Current data files with column-level stats |
| `"table$manifests"` | Manifest files that group data files |
| `"table$partitions"` | Partition-level record/file counts |
| `"table$refs"` | Named references (branches / tags) |

**Prerequisites:** Docker Compose stack must be running (`docker compose up -d`)  
Trino coordinator is available at `http://trino-coordinator:8080`

---
## 0. Setup: connect to Trino
---

In [1]:
import trino
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

conn = trino.dbapi.connect(
    host="trino-coordinator",
    port=8080,
    user="jupyter",
    catalog="iceberg",
    schema="demo",
)

def query(sql: str) -> pd.DataFrame:
    """Run a Trino SQL query and return a DataFrame."""
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description] if cur.description else []
    return pd.DataFrame(rows, columns=cols)

def meta(table: str, suffix: str) -> str:
    """Return the fully-qualified Trino metadata table reference.
    
    Trino requires the table name and $suffix to be quoted together as a
    single identifier, e.g.  iceberg.demo."transactions$history"
    """
    # table is just the bare name, e.g. 'transactions'
    return f'iceberg.demo."{table}${suffix}"'

# Verify connection and list available tables
print("Connected to Trino. Tables in iceberg.demo:")
query("SHOW TABLES IN iceberg.demo")

Connected to Trino. Tables in iceberg.demo:


,Table
0,transactions
1,users


---
## 1. Select a table
---

Choose which Iceberg table to inspect. All metadata sections below will use the selected table.

In [2]:
available_tables = ["transactions", "users"]

table_selector = widgets.Dropdown(
    options=available_tables,
    value=available_tables[0],
    description="Table:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)

display(table_selector)

Dropdown(description='Table:', layout=Layout(width='400px'), options=('transactions', 'users'), style=Descript…

In [3]:
# Re-run this cell after changing the dropdown above
TABLE = table_selector.value
FULL_TABLE = f"iceberg.demo.{TABLE}"
print(f"Inspecting: {FULL_TABLE}")

Inspecting: iceberg.demo.transactions


---
## 2. Select table Properties
---
Table properies store basic Iceberg-related configration settings on the table level

In [12]:
# show the table properties
query(f"""
SELECT * FROM {meta(TABLE, 'properties')}
"""
)

,key,value
0,created-at,2026-05-15T17:56:58.933022172Z
1,write.format.default,parquet
2,write.metadata.previous-versions-max,2147483647
3,write.parquet.compression-codec,zstd
4,write.upsert.enabled,false


---
## 3. History (`$history`)
---


The `$history` metadata table tracks every snapshot committed to the table.

| Column | Type | Description |
|---|---|---|
| `made_current_at` | timestamp | When this snapshot became current |
| `snapshot_id` | bigint | Unique snapshot identifier |
| `parent_id` | bigint | Parent snapshot ID (null for first) |
| `is_current_ancestor` | boolean | On the current lineage chain? |

The [$history](https://trino.io/docs/current/connector/iceberg.html#history-table) table provides a log of the metadata changes performed on the Iceberg table. 

You can retrieve the changelog of the Iceberg table test_table by using the following query:

In [14]:
history_df = query(f"""
    SELECT
         *
    FROM {meta(TABLE, 'history')}

""")

print(f"Total snapshots in history: {len(history_df)}")
history_df

Total snapshots in history: 12


,made_current_at,snapshot_id,parent_id,is_current_ancestor
0,2026-05-15 17:57:13.090000+00:00,7911153381993429235,NaN,True
1,2026-05-15 17:57:39.608000+00:00,2420435701824803222,7.911153e+18,True
2,2026-05-15 17:58:10.022000+00:00,4377751641787504921,2.420436e+18,True
3,2026-05-15 17:58:39.612000+00:00,1339610066997163940,4.377752e+18,True
4,2026-05-15 17:59:09.685000+00:00,5434184491940196438,1.339610e+18,True
5,2026-05-15 17:59:39.702000+00:00,4556900777407928834,5.434184e+18,True
6,2026-05-15 18:00:09.661000+00:00,2502484812322068064,4.556901e+18,True
7,2026-05-15 18:00:39.619000+00:00,7867871634301297068,2.502485e+18,True
8,2026-05-15 18:01:09.698000+00:00,8915160460029123347,7.867872e+18,True
9,2026-05-15 18:01:39.801000+00:00,3587989789569619766,8.915160e+18,True


## 3.1 time travel with snapshot id
Given the snapshot id we can query the table as of specific time/snapshot, practically speaking making a time travel.

In [17]:
# Time-travel: read the table AS OF a specific snapshot
if not history_df.empty:
    oldest_snapshot_id = history_df["snapshot_id"].iloc[-1]
    print(f"Reading {FULL_TABLE} AS OF snapshot {oldest_snapshot_id} (oldest):")
    display(query(f"""
        SELECT *
        FROM {FULL_TABLE}
        FOR VERSION AS OF {oldest_snapshot_id}
        LIMIT 5
    """))
else:
    print("No history found.")

Reading iceberg.demo.transactions AS OF snapshot 5024575875782902844 (oldest):


,transaction_id,user_id,amount,currency,type,status,event_time
0,txn-000754,user-011,21.47,USD,PURCHASE,COMPLETED,2026-05-15 18:02:09.649
1,txn-000755,user-024,1156.81,GBP,TRANSFER,FAILED,2026-05-15 18:02:10.151
2,txn-000756,user-019,1659.11,AUD,DEPOSIT,REVERSED,2026-05-15 18:02:10.652
3,txn-000757,user-027,1090.33,AUD,PURCHASE,COMPLETED,2026-05-15 18:02:11.153
4,txn-000758,user-008,1497.91,CAD,PURCHASE,REVERSED,2026-05-15 18:02:11.656


---
## 4. Snapshots (`$snapshots`)

The `$snapshots` table reveals the **content** of each snapshot — operation type, manifest counts, and record/file deltas via the `summary` map.

| Column | Type | Description |
|---|---|---|
| `committed_at` | timestamp | Wall-clock time of the commit |
| `snapshot_id` | bigint | Unique snapshot ID |
| `parent_id` | bigint | Parent snapshot ID |
| `operation` | string | `append`, `overwrite`, `replace`, `delete` |
| `manifest_list` | string | Path to the manifest-list Avro file |
| `summary` | map | Key metrics: added/deleted records, data files |

In [19]:
# All snapshots — most recent first
snapshots_df = query(f"""
    SELECT
        *
    FROM {meta(TABLE, 'snapshots')}
    ORDER BY committed_at DESC
""")

print(f"Snapshots: {len(snapshots_df)}")
snapshots_df

Snapshots: 24


,committed_at,snapshot_id,parent_id,operation,manifest_list,summary
0,2026-05-15 18:08:39.580000+00:00,729683183671619992,9.171082e+18,append,s3://iceberg-warehouse/demo/transactions/metad...,{'flink.operator-id': 'cf155f65686cb012844f7c7...
1,2026-05-15 18:08:09.643000+00:00,9171082188076313309,4.008629e+18,append,s3://iceberg-warehouse/demo/transactions/metad...,{'flink.operator-id': 'cf155f65686cb012844f7c7...
2,2026-05-15 18:07:39.637000+00:00,4008629299437918640,1.201339e+18,append,s3://iceberg-warehouse/demo/transactions/metad...,{'flink.operator-id': 'cf155f65686cb012844f7c7...
3,2026-05-15 18:07:09.633000+00:00,1201339357889859424,8.593879e+18,append,s3://iceberg-warehouse/demo/transactions/metad...,{'flink.operator-id': 'cf155f65686cb012844f7c7...
4,2026-05-15 18:06:39.705000+00:00,8593879230531432674,8.829209e+18,append,s3://iceberg-warehouse/demo/transactions/metad...,{'flink.operator-id': 'cf155f65686cb012844f7c7...
5,2026-05-15 18:06:09.661000+00:00,8829208564651232998,1.616662e+18,append,s3://iceberg-warehouse/demo/transactions/metad...,{'flink.operator-id': 'cf155f65686cb012844f7c7...
6,2026-05-15 18:05:39.637000+00:00,1616662076194259502,1.719432e+18,append,s3://iceberg-warehouse/demo/transactions/metad...,{'flink.operator-id': 'cf155f65686cb012844f7c7...
7,2026-05-15 18:05:09.739000+00:00,1719432404622747749,5.087711e+18,append,s3://iceberg-warehouse/demo/transactions/metad...,{'flink.operator-id': 'cf155f65686cb012844f7c7...
8,2026-05-15 18:04:39.571000+00:00,5087710550209431726,2.377479e+18,append,s3://iceberg-warehouse/demo/transactions/metad...,{'flink.operator-id': 'cf155f65686cb012844f7c7...
9,2026-05-15 18:04:09.938000+00:00,2377479493711205715,3.007103e+18,append,s3://iceberg-warehouse/demo/transactions/metad...,{'flink.operator-id': 'cf155f65686cb012844f7c7...


In [20]:
# Extract key metrics from the summary map
# summary is a map<varchar,varchar> — use element_at() to pull out individual keys
query(f"""
    SELECT
        committed_at,
        snapshot_id,
        operation,
        CAST(element_at(summary, 'added-records')         AS bigint) AS added_records,
        CAST(element_at(summary, 'deleted-records')       AS bigint) AS deleted_records,
        CAST(element_at(summary, 'added-data-files')      AS bigint) AS added_data_files,
        CAST(element_at(summary, 'removed-data-files')    AS bigint) AS removed_data_files,
        CAST(element_at(summary, 'total-records')         AS bigint) AS total_records,
        CAST(element_at(summary, 'total-data-files')      AS bigint) AS total_data_files
    FROM {meta(TABLE, 'snapshots')}
    ORDER BY committed_at DESC
""")

,committed_at,snapshot_id,operation,added_records,deleted_records,added_data_files,removed_data_files,total_records,total_data_files
0,2026-05-15 18:10:39.856000+00:00,3028053191347306034,append,60,None,1,None,45799,28
1,2026-05-15 18:10:09.607000+00:00,6380888278381790282,append,59,None,1,None,45739,27
2,2026-05-15 18:09:39.594000+00:00,5188079848495828535,append,60,None,1,None,45680,26
3,2026-05-15 18:09:09.580000+00:00,1390970797981214610,append,59,None,1,None,45620,25
4,2026-05-15 18:08:39.580000+00:00,729683183671619992,append,60,None,1,None,45561,24
5,2026-05-15 18:08:09.643000+00:00,9171082188076313309,append,60,None,1,None,45501,23
6,2026-05-15 18:07:39.637000+00:00,4008629299437918640,append,59,None,1,None,45441,22
7,2026-05-15 18:07:09.633000+00:00,1201339357889859424,append,60,None,1,None,45382,21
8,2026-05-15 18:06:39.705000+00:00,8593879230531432674,append,60,None,1,None,45322,20
9,2026-05-15 18:06:09.661000+00:00,8829208564651232998,append,59,None,1,None,45262,19


---
## 5. Files (`$files`)
---

The `$files` metadata table lists every **current data file** with column-level statistics. This is the key table for diagnosing data skew, small-file problems, and partition layout.

| Column | Type | Description |
|---|---|---|
| `content` | int | 0 = DATA, 1 = POSITION_DELETES, 2 = EQUALITY_DELETES |
| `file_path` | string | Full S3 path |
| `file_format` | string | `PARQUET`, `ORC`, `AVRO` |
| `record_count` | bigint | Rows in this file |
| `file_size_in_bytes` | bigint | Compressed file size |
| `column_sizes` | map | Bytes used per column field ID |
| `value_counts` | map | Non-null values per field ID |
| `null_value_counts` | map | Null values per field ID |
| `lower_bounds` | map | Min value per field ID |
| `upper_bounds` | map | Max value per field ID |

## 5.1 Let's show a sample of file level stats

In [25]:
# All current data files — most rows first
files_df = query(f"""
    SELECT
         *
    FROM {meta(TABLE, 'files')}
    ORDER BY record_count DESC
""")

print(f"Total data files: {len(files_df)}")
files_df[:5]
# files_df[["content", "file_format", "record_count", "file_size_in_bytes", "file_path"]]

Total data files: 35


,content,file_path,file_format,spec_id,record_count,file_size_in_bytes,column_sizes,value_counts,null_value_counts,nan_value_counts,lower_bounds,upper_bounds,key_metadata,split_offsets,equality_ids,sort_order_id,readable_metrics
0,0,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,0,44189,374847,"{1: 17470, 2: 33543, 3: 148310, 4: 15247, 5: 1...","{1: 44189, 2: 44189, 3: 44189, 4: 44189, 5: 44...","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0}",{3: 0},"{1: 'txn-000001', 2: 'user-001', 3: '1.1', 4: ...","{1: 'txn-044037', 2: 'user-050', 3: '2000.0', ...",None,[4],None,0,"{""amount"":{""column_size"":148310,""value_count"":..."
1,0,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,0,60,3304,"{1: 184, 2: 228, 3: 393, 4: 133, 5: 158, 6: 13...","{1: 60, 2: 60, 3: 60, 4: 60, 5: 60, 6: 60, 7: 60}","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0}",{3: 0},"{1: 'txn-002006', 2: 'user-001', 3: '6.53', 4:...","{1: 'txn-002065', 2: 'user-048', 3: '1987.41',...",None,[4],None,0,"{""amount"":{""column_size"":393,""value_count"":60,..."
2,0,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,0,60,3305,"{1: 183, 2: 225, 3: 398, 4: 133, 5: 158, 6: 13...","{1: 60, 2: 60, 3: 60, 4: 60, 5: 60, 6: 60, 7: 60}","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0}",{3: 0},"{1: 'txn-002066', 2: 'user-002', 3: '39.91', 4...","{1: 'txn-002125', 2: 'user-050', 3: '1996.64',...",None,[4],None,0,"{""amount"":{""column_size"":398,""value_count"":60,..."
3,0,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,0,60,3301,"{1: 184, 2: 228, 3: 391, 4: 133, 5: 157, 6: 13...","{1: 60, 2: 60, 3: 60, 4: 60, 5: 60, 6: 60, 7: 60}","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0}",{3: 0},"{1: 'txn-001112', 2: 'user-001', 3: '37.48', 4...","{1: 'txn-001171', 2: 'user-050', 3: '1940.48',...",None,[4],None,0,"{""amount"":{""column_size"":391,""value_count"":60,..."
4,0,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,0,60,3306,"{1: 183, 2: 238, 3: 387, 4: 133, 5: 158, 6: 13...","{1: 60, 2: 60, 3: 60, 4: 60, 5: 60, 6: 60, 7: 60}","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0}",{3: 0},"{1: 'txn-001052', 2: 'user-001', 3: '107.15', ...","{1: 'txn-001111', 2: 'user-050', 3: '1997.31',...",None,[4],None,0,"{""amount"":{""column_size"":387,""value_count"":60,..."


## 5.2 Let's check if we have a small files issue

In [26]:
# File-level size statistics — spot small-file problems
query(f"""
    SELECT
        file_format,
        count(*)                                            AS file_count,
        sum(record_count)                                   AS total_records,
        round(avg(record_count), 0)                         AS avg_records_per_file,
        min(record_count)                                   AS min_records,
        max(record_count)                                   AS max_records,
        sum(file_size_in_bytes)                             AS total_size_bytes,
        round(avg(file_size_in_bytes) / 1024.0 / 1024, 2)  AS avg_size_mb
    FROM {meta(TABLE, 'files')}
    GROUP BY file_format
""")

,file_format,file_count,total_records,avg_records_per_file,min_records,max_records,total_size_bytes,avg_size_mb
0,PARQUET,36,46276,1285.0,58,44189,490291,0.01


---
## 6. Metadata log entries
---
The $metadata_log_entries table provides a view of metadata log entries of the Iceberg table.

You can retrieve the information about the metadata log entries of the Iceberg table test_table

by using the following query:

In [45]:
# The manifests for the current snapshot
manifests_df = query(f"""
    SELECT
        *
    FROM {meta(TABLE, 'metadata_log_entries')}
""")

print(f"Total manifests: {len(manifests_df)}")
manifests_df[:5]

Total manifests: 72


,timestamp,file,latest_snapshot_id,latest_schema_id,latest_sequence_number
0,2026-05-15 17:56:58.981000+00:00,s3://iceberg-warehouse/demo/transactions/metad...,NaN,NaN,NaN
1,2026-05-15 17:57:13.090000+00:00,s3://iceberg-warehouse/demo/transactions/metad...,7.911153e+18,0.0,1.0
2,2026-05-15 17:57:39.608000+00:00,s3://iceberg-warehouse/demo/transactions/metad...,2.420436e+18,0.0,2.0
3,2026-05-15 17:58:10.022000+00:00,s3://iceberg-warehouse/demo/transactions/metad...,4.377752e+18,0.0,3.0
4,2026-05-15 17:58:39.612000+00:00,s3://iceberg-warehouse/demo/transactions/metad...,1.339610e+18,0.0,4.0


---
## 7. Manifests (`$manifests`)
---

Manifest files are Avro files that group data files. The `$manifests` table exposes every manifest referenced by the current snapshot, including partition-level stats that let Iceberg prune manifests at planning time.

| Column | Type | Description |
|---|---|---|
| `path` | string | S3 path to the Avro manifest |
| `length` | bigint | Manifest file size in bytes |
| `partition_spec_id` | int | Partition spec used to write these files |
| `added_snapshot_id` | bigint | Snapshot that added this manifest |
| `added_data_files_count` | int | Data files added in this manifest |
| `existing_data_files_count` | int | Unchanged data files |
| `deleted_data_files_count` | int | Logically deleted files pending cleanup |
| `added_rows_count` | bigint | Rows added |
| `existing_rows_count` | bigint | Unchanged rows |
| `deleted_rows_count` | bigint | Deleted rows not yet compacted |

In [28]:
# The manifests for the current snapshot
manifests_df = query(f"""
    SELECT
        *
    FROM {meta(TABLE, 'manifests')}
    ORDER BY added_snapshot_id DESC
""")

print(f"Total manifests: {len(manifests_df)}")
manifests_df[:5]

Total manifests: 40


,path,length,partition_spec_id,added_snapshot_id,added_data_files_count,added_rows_count,existing_data_files_count,existing_rows_count,deleted_data_files_count,deleted_rows_count,partition_summaries
0,s3://iceberg-warehouse/demo/transactions/metad...,7487,0,9171082188076313309,1,60,0,0,0,0,[]
1,s3://iceberg-warehouse/demo/transactions/metad...,7487,0,8915160460029123347,1,59,0,0,0,0,[]
2,s3://iceberg-warehouse/demo/transactions/metad...,7487,0,8829208564651232998,1,59,0,0,0,0,[]
3,s3://iceberg-warehouse/demo/transactions/metad...,7488,0,8732771105717539472,1,60,0,0,0,0,[]
4,s3://iceberg-warehouse/demo/transactions/metad...,7488,0,8593879230531432674,1,60,0,0,0,0,[]


In [44]:
# Manifest health — pending deletes indicate compaction is needed
query(f"""
    SELECT
        count(*)                        AS manifest_count,
        sum(added_data_files_count)     AS total_added_files,
        sum(existing_data_files_count)  AS total_existing_files,
        sum(deleted_data_files_count)   AS total_pending_deleted_files,
        sum(added_rows_count)           AS total_added_rows,
        sum(existing_rows_count)        AS total_existing_rows,
        sum(deleted_rows_count)         AS total_pending_deleted_rows
    FROM {meta(TABLE, 'manifests')}
""")

,manifest_count,total_added_files,total_existing_files,total_pending_deleted_files,total_added_rows,total_existing_rows,total_pending_deleted_rows
0,10,10,198,0,597,15213,0


## 7.1 `all_manifests` vs `manifests`

The `$manifests` and `$all_manifests` tables provide a detailed overview of the manifests corresponding to
the snapshots performed in the log of the Iceberg table.

The `$manifests` table contains data for the current snapshot.
The `$all_manifests` table contains data for all snapshots.

In [43]:
# The all_manifests for the current snapshot
manifests_df = query(f"""
    SELECT
        *
    FROM {meta(TABLE, 'all_manifests')}
    ORDER BY added_snapshot_id DESC
""")

print(f"Total manifests: {len(manifests_df)}")
manifests_df[:5]

Total manifests: 2346


,path,length,partition_spec_id,added_snapshot_id,added_data_files_count,existing_data_files_count,deleted_data_files_count,partition_summaries
0,s3://iceberg-warehouse/demo/transactions/metad...,7490,0,9184143483856690979,1,0,0,[]
1,s3://iceberg-warehouse/demo/transactions/metad...,7490,0,9184143483856690979,1,0,0,[]
2,s3://iceberg-warehouse/demo/transactions/metad...,7490,0,9184143483856690979,1,0,0,[]
3,s3://iceberg-warehouse/demo/transactions/metad...,7487,0,9171082188076313309,1,0,0,[]
4,s3://iceberg-warehouse/demo/transactions/metad...,7487,0,9171082188076313309,1,0,0,[]


---
## 8. Partitions (`$partitions`)
---

The `$partitions` table provides a **partition-level summary** — record counts, file counts, and value bounds rolled up per partition value. Essential for spotting data skew.

| Column | Type | Description |
|---|---|---|
| `partition` | row | Partition column values |
| `record_count` | bigint | Total rows in this partition |
| `file_count` | bigint | Data files in this partition |
| `total_size` | bigint | Total compressed bytes |
| `data` | map | Per-column stats (null_count, nan_count, lower_bound, upper_bound) |

> If the table is unpartitioned, `$partitions` returns a single row.

In [29]:
# Partition-level stats
# For unpartitioned tables the 'partition' column is empty/absent — use SELECT * to get whatever columns exist
partitions_df = query(f"SELECT * FROM {meta(TABLE, 'partitions')}")

print(f"Columns : {list(partitions_df.columns)}")
print(f"Partitions: {len(partitions_df)}")
partitions_df

Columns : ['record_count', 'file_count', 'total_size', 'data']
Partitions: 1


,record_count,file_count,total_size,data
0,46634,42,510085,"(transaction_id: (min: 'txn-000001', max: 'txn..."


In [30]:
# Skew detection across partitions
if len(partitions_df) > 1:
    skew_df = partitions_df["record_count"].describe().to_frame()
    print("Record count distribution across partitions:")
    display(skew_df)
    skew_ratio = partitions_df["record_count"].max() / partitions_df["record_count"].min()
    print(f"\nMax/Min skew ratio: {skew_ratio:.1f}x")
    if skew_ratio > 10:
        print("WARNING: High data skew detected — consider repartitioning.")
    else:
        print("Partition distribution looks balanced.")
else:
    print("Table is unpartitioned or has a single partition.")

Table is unpartitioned or has a single partition.


---
## 9. Refs (`$refs`)
---

The `$refs` table shows all named references (branches and tags) for this table. With Apache Polaris, only the default `main` branch exists — Git-like multi-branch branching is a Nessie-specific feature.

| Column | Type | Description |
|---|---|---|
| `name` | string | Reference name (`main`) |
| `type` | string | `BRANCH` or `TAG` |
| `snapshot_id` | bigint | Snapshot this reference points to |
| `max_reference_age_in_ms` | bigint | Retention policy |
| `min_snapshots_to_keep` | int | Minimum snapshots to retain |
| `max_snapshot_age_in_ms` | bigint | Snapshot age expiry threshold |

In [31]:
# Named references (branches / tags) via Iceberg $refs
refs_df = query(f"""
    SELECT
        *
    FROM {meta(TABLE, 'refs')}
""")

print(f"References (branches / tags): {len(refs_df)}")
refs_df

References (branches / tags): 1


,name,type,snapshot_id,max_reference_age_in_ms,min_snapshots_to_keep,max_snapshot_age_in_ms
0,main,BRANCH,1343923430394415687,None,None,None


---
## 10. Sample data & basic statistics
---

Quick sanity-check: read some rows and compute summary stats directly via Trino SQL.

In [32]:
# Sample 10 rows from the table
print(f"Sample rows from {FULL_TABLE}:")
query(f"SELECT * FROM {FULL_TABLE} LIMIT 10")

Sample rows from iceberg.demo.transactions:


,transaction_id,user_id,amount,currency,type,status,event_time
0,txn-000396,user-003,1223.63,GBP,REFUND,COMPLETED,2026-05-15 17:59:09.707
1,txn-000397,user-010,668.20,AUD,TRANSFER,FAILED,2026-05-15 17:59:10.210
2,txn-000398,user-041,361.94,CAD,TRANSFER,REVERSED,2026-05-15 17:59:10.713
3,txn-000399,user-018,1149.96,CAD,PURCHASE,REVERSED,2026-05-15 17:59:11.218
4,txn-000400,user-048,988.04,AUD,WITHDRAWAL,REVERSED,2026-05-15 17:59:11.720
5,txn-000401,user-047,1289.43,EUR,TRANSFER,FAILED,2026-05-15 17:59:12.223
6,txn-000402,user-023,744.23,AUD,DEPOSIT,COMPLETED,2026-05-15 17:59:12.725
7,txn-000403,user-045,1659.65,AUD,DEPOSIT,REVERSED,2026-05-15 17:59:13.228
8,txn-000404,user-010,971.46,EUR,WITHDRAWAL,REVERSED,2026-05-15 17:59:13.733
9,txn-000405,user-026,266.07,GBP,DEPOSIT,FAILED,2026-05-15 17:59:14.234


In [33]:
# Table-specific statistics — transactions
if TABLE == "transactions":
    print("Transactions — summary statistics:")
    display(query(f"""
        SELECT
            count(*)                        AS total_rows,
            count(DISTINCT user_id)         AS distinct_users,
            round(avg(amount), 2)           AS avg_amount,
            min(amount)                     AS min_amount,
            max(amount)                     AS max_amount,
            min(event_time)                 AS earliest_event,
            max(event_time)                 AS latest_event
        FROM {FULL_TABLE}
    """))

    print("\nBreakdown by status:")
    display(query(f"""
        SELECT
            status,
            count(*)                    AS txn_count,
            round(sum(amount), 2)       AS total_amount
        FROM {FULL_TABLE}
        GROUP BY status
        ORDER BY txn_count DESC
    """))

    print("\nBreakdown by type:")
    display(query(f"""
        SELECT
            type,
            count(*)                    AS txn_count,
            round(avg(amount), 2)       AS avg_amount
        FROM {FULL_TABLE}
        GROUP BY type
        ORDER BY txn_count DESC
    """))

elif TABLE == "users":
    print("Users — summary statistics:")
    display(query(f"""
        SELECT
            count(*)                    AS total_users,
            count(DISTINCT country)     AS distinct_countries,
            min(created_at)             AS first_user,
            max(created_at)             AS latest_user
        FROM {FULL_TABLE}
    """))

    print("\nTop 10 countries by user count:")
    display(query(f"""
        SELECT
            country,
            count(*) AS user_count
        FROM {FULL_TABLE}
        GROUP BY country
        ORDER BY user_count DESC
        LIMIT 10
    """))

Transactions — summary statistics:


,total_rows,distinct_users,avg_amount,min_amount,max_amount,earliest_event,latest_event
0,47052,50,997.52,1.1,2000.0,2026-05-14 18:26:55.065,2026-05-15 18:21:08.939



Breakdown by status:


,status,txn_count,total_amount
0,PENDING,11968,11976361.90
1,REVERSED,11779,11708058.90
2,FAILED,11683,11716850.64
3,COMPLETED,11622,11533830.75



Breakdown by type:


,type,txn_count,avg_amount
0,PURCHASE,9536,996.87
1,REFUND,9448,1002.57
2,WITHDRAWAL,9446,993.31
3,DEPOSIT,9330,992.85
4,TRANSFER,9292,1002.00


---
## 11. Table health dashboard
---

Combine all metadata sources into a single health summary — snapshot count, file sizes, manifest bloat, and pending deletes.

In [37]:
# ── Snapshot count ──────────────────────────────────────────────────────────
snap_row = query(f"SELECT count(*) AS cnt FROM {meta(TABLE, 'history')}").iloc[0]
snapshot_count = snap_row["cnt"]

# ── File stats ──────────────────────────────────────────────────────────────
file_row = query(f"""
    SELECT
        count(*)                                            AS file_count,
        sum(record_count)                                   AS total_records,
        round(avg(record_count), 0)                         AS avg_records_per_file,
        round(avg(file_size_in_bytes) / 1024.0 / 1024, 2)  AS avg_size_mb,
        sum(file_size_in_bytes)                             AS total_size_bytes
    FROM {meta(TABLE, 'files')}
    WHERE content = 0
""").iloc[0]

# ── Manifest bloat ──────────────────────────────────────────────────────────
manifest_row = query(f"""
    SELECT
        count(*)                        AS manifest_count,
        sum(deleted_data_files_count)   AS pending_deleted_files,
        sum(deleted_rows_count)         AS pending_deleted_rows
    FROM {meta(TABLE, 'manifests')}
""").iloc[0]

# ── Print summary ───────────────────────────────────────────────────────────
SEP = "=" * 56
print(SEP)
print(f"  TABLE HEALTH: {FULL_TABLE}")
print(SEP)
print(f"  Snapshots committed        : {snapshot_count}")
print(f"  Current data files         : {file_row['file_count']}")
print(f"  Total records              : {int(file_row['total_records'] or 0):,}")
print(f"  Avg records per file       : {int(file_row['avg_records_per_file'] or 0):,}")
print(f"  Avg file size              : {file_row['avg_size_mb']} MB")
print(f"  Total data size            : {round((file_row['total_size_bytes'] or 0) / 1024 / 1024, 2)} MB")
print(f"  Manifests                  : {manifest_row['manifest_count']}")
print(f"  Pending deleted files      : {manifest_row['pending_deleted_files'] or 0}")
print(f"  Pending deleted rows       : {int(manifest_row['pending_deleted_rows'] or 0):,}")
print()
print("  Recommendations:")
avg_recs = int(file_row["avg_records_per_file"] or 0)
if avg_recs < 10_000:
    print("  [!] Low records-per-file — consider running RewriteDataFiles compaction")
else:
    print("  [ok] File sizes look healthy")
pending_del = int(manifest_row["pending_deleted_files"] or 0)
if pending_del > 0:
    print("  [!] Pending deletes — run expireSnapshots to reclaim storage")
else:
    print("  [ok] No pending deletes")
print(SEP)

  TABLE HEALTH: iceberg.demo.transactions
  Snapshots committed        : 51
  Current data files         : 51.0
  Total records              : 47,172
  Avg records per file       : 925
  Avg file size              : 0.01 MB
  Total data size            : 0.51 MB
  Manifests                  : 51
  Pending deleted files      : 0
  Pending deleted rows       : 0

  Recommendations:
  [!] Low records-per-file — consider running RewriteDataFiles compaction
  [ok] No pending deletes


---
## 12 Trino cluster info
---

Inspect the Trino cluster itself: active workers, running queries, catalog configuration.

In [38]:
import urllib.request, json

# Trino REST API — cluster info
with urllib.request.urlopen("http://trino-coordinator:8080/v1/info") as r:
    info = json.loads(r.read())
print("Trino cluster info:")
for k, v in info.items():
    print(f"  {k}: {v}")

Trino cluster info:
  nodeVersion: {'version': '468'}
  environment: production
  coordinator: True
  starting: False
  uptime: 30.43m


In [39]:
# Active worker nodes
# /v1/node requires the X-Trino-User header (unlike /v1/info which is public)
req = urllib.request.Request(
    "http://trino-coordinator:8080/v1/node",
    headers={"X-Trino-User": "jupyter"},
)
with urllib.request.urlopen(req) as r:
    nodes = json.loads(r.read())

nodes_df = pd.DataFrame([
    {
        "node_id": n.get("nodeId"),
        "uri": n.get("uri"),
        "state": n.get("state"),
        "coordinator": n.get("coordinator"),
    }
    for n in nodes
])
print(f"Active nodes: {len(nodes_df)}")
nodes_df

Active nodes: 1


,node_id,uri,state,coordinator
0,None,http://172.20.0.12:8080/v1/status,None,None


In [40]:
# Available catalogs
print("Catalogs available in Trino:")
query("SHOW CATALOGS")

Catalogs available in Trino:


,Catalog
0,iceberg
1,jmx
2,memory
3,system
4,tpcds
5,tpch


In [41]:
# Schemas in the iceberg catalog
print("Schemas in the iceberg catalog:")
query("SHOW SCHEMAS IN iceberg")

Schemas in the iceberg catalog:


,Schema
0,demo
1,information_schema
